Installing CodeCarbon

In [1]:
!pip install -q codecarbon

Magnitude Pruning Step 4: Pruned Model Results and Evaluation

In [2]:
import os
import csv
import glob
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from codecarbon import OfflineEmissionsTracker

model_name = "codellama/CodeLlama-7b-Instruct-hf"   # tokenizer source (pruning never changes it)
model_dir = "/content/pruned_model"
version = "pruned_40"
country = "USA"        # must match the other three scripts
num_runs = 5
max_new_tokens = 300

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")
hardware = torch.cuda.get_device_name(0)

# Load only if this exact model + version isn't already in memory
if globals().get("loaded_config") != (model_dir, version):
    if "model" in globals():
        raise RuntimeError("A different model/version is loaded. Restart the session first.")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.float16, device_map="cuda")
    model.eval()
    loaded_config = (model_dir, version)
else:
    print("Model already loaded, skipping")

# Model size = storage size of the saved weights (dense format: zeros are still stored)
size_gb = sum(os.path.getsize(f) for f in glob.glob(os.path.join(model_dir, "*.safetensors"))) / 1e9

# Sparsity, and the theoretical size if the pruned layers were stored in a sparse format:
# non-zero values (2 bytes each, FP16) + a 1-bit mask per weight; other parameters stay dense.
pruned_total, zero_total = 0, 0
for layer in model.model.layers:
    for module in layer.modules():
        if isinstance(module, torch.nn.Linear):
            pruned_total += module.weight.numel()
            zero_total += (module.weight == 0).sum().item()
all_params = sum(p.numel() for p in model.parameters())
dense_params = all_params - pruned_total
sparse_bytes = (pruned_total - zero_total) * 2 + pruned_total / 8 + dense_params * 2
sparse_gb = sparse_bytes / 1e9

print("Version:", version)
print("Hardware:", hardware)
print("Dtype:", model.dtype)
print(f"Sparsity in pruned layers: {zero_total / pruned_total:.2%}")
print(f"Model size on disk (dense format): {size_gb:.2f} GB")
print(f"Theoretical size in sparse format (values + bitmask): {sparse_gb:.2f} GB")
print(f"GPU memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

Code = '''from typing import List
def below_zero(operations: List[int]) -> bool:
    """ You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False'''

prompt = ("Generate Google-style docstring for the following Python function: {code}. "
          "Only output the docstring and function code.").format(code=Code)   # P-2

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(model.device)
input_tokens = inputs["input_ids"].shape[1]

results_csv = f"results_{version}.csv"
outputs_file = f"docstrings_{version}.txt"

# Warm up once (untimed, untracked)
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)

with open(results_csv, "w", newline="") as f, open(outputs_file, "w") as out_f:
    writer = csv.writer(f)
    writer.writerow(["run", "input_tokens", "output_tokens", "latency_s",
                     "emissions_kg", "energy_kwh", "accuracy_PF"])

    for run in range(1, num_runs + 1):
        # Fresh tracker per run, wrapped around generation only
        tracker = OfflineEmissionsTracker(country_iso_code=country, measure_power_secs=1,
                                          save_to_file=False, log_level="error")
        tracker.start()
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                           do_sample=True, temperature=0.7,
                                           eos_token_id=tokenizer.eos_token_id,
                                           pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        latency = time.perf_counter() - start
        emissions_kg = tracker.stop()
        energy_kwh = tracker.final_emissions_data.energy_consumed

        output_tokens = generated_ids.shape[1] - input_tokens
        text = tokenizer.decode(generated_ids[0][input_tokens:], skip_special_tokens=True)
        out_f.write(f"--- Run {run} ---\n{text}\n\n")

        writer.writerow([run, input_tokens, output_tokens, f"{latency:.3f}",
                         f"{emissions_kg:.4e}", f"{energy_kwh:.4e}", ""])
        f.flush()

        print(f"Run {run}: {latency:.3f}s | {output_tokens} tokens | "
              f"{emissions_kg:.4e} kg CO2eq | {energy_kwh:.4e} kWh")

print(f"Docstrings saved to {outputs_file}")
print(f"Metrics saved to {results_csv}")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Version: pruned_40
Hardware: Tesla T4
Dtype: torch.float16
Sparsity in pruned layers: 40.08%
Model size on disk (dense format): 13.48 GB
Theoretical size in sparse format (values + bitmask): 9.10 GB
GPU memory footprint: 13.48 GB


[codecarbon INFO @ 20:00:58] offline tracker init
[codecarbon WARNING @ 20:00:58] Multiple instances of codecarbon are allowed to run at the same time.


Run 1: 9.801s | 158 tokens | 7.9933e-05 kg CO2eq | 2.1634e-04 kWh
Run 2: 7.462s | 121 tokens | 6.0997e-05 kg CO2eq | 1.6509e-04 kWh
Run 3: 7.244s | 115 tokens | 5.8835e-05 kg CO2eq | 1.5924e-04 kWh
Run 4: 7.569s | 115 tokens | 6.0987e-05 kg CO2eq | 1.6506e-04 kWh
Run 5: 11.646s | 115 tokens | 9.2002e-05 kg CO2eq | 2.4901e-04 kWh
Docstrings saved to docstrings_pruned_40.txt
Metrics saved to results_pruned_40.csv


Downloading Files

In [3]:
from google.colab import files

files.download(results_csv)
files.download(outputs_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>